In [ ]:
import pulp
from pulp import LpVariable ,lpSum
import pandas as pd
from typing import cast


df = pd.read_csv("data/students.csv")
df_pair = pd.read_csv("data/student_pairs.csv")


classSize = 8
studentSize= df["student_id"].tolist()
students = df["student_id"].tolist()
classes = range(1,classSize+1)

prob = pulp.LpProblem(name="ClassDivide")
a = LpVariable.dicts("classNum",(df.student_id,classes) ,cat="Binary")

prob += 0

for i in classes:
    prob += lpSum(a[s][i] for s in studentSize) <= 40
    prob += lpSum(a[s][i] for s in studentSize) >= 39
    
for i in studentSize:
    prob += lpSum(a[i][c] for c in classes) == 1

male = df.loc[df["gender"] == 1 ,"student_id"].tolist()
female = df.loc[df["gender"] == 0, "student_id"].tolist()

for c in classes:
    prob += lpSum(a[m][c] for m in male) <= 20
    
for c in classes:
    prob += lpSum(a[f][c] for f in female) <= 20
    
score = dict(zip(df["student_id"], df["score"]))
average = df["score"].mean()

for c in classes:
    class_score = lpSum(score[s] * a[s][c] for s in students)
    class_count = lpSum(a[s][c] for s in students)

    prob += class_score - average * class_count <= 10 * class_count
    prob += class_score - average * class_count >= -10 * class_count

leader = df.loc[df["leader_flag"] == 1 ,"student_id"].tolist()
for c in classes:
    prob += lpSum(a[l][c] for l in leader) >= 2
    
support = df.loc[df["support_flag"] == 1 ,"student_id"].tolist()
for c in classes:
    prob += lpSum(a[s][c] for s in support) <= 1
    

for _, row in df_pair.iterrows():
    s1 = row["student_id1"]
    s2 = row["student_id2"]

    for c in classes:
        prob += a[s1][c] + a[s2][c] <= 1

status =  cast(int, prob.solve())


print(status)
print("Status:", pulp.LpStatus[status])


for i in students:
    for j in classes:
        if a[i][j] == 1:
            print("student ",i,"class is:",j)
            
            

Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /home/hayato/university/.venv/lib/python3.12/site-packages/pulp/apis/../solverdir/cbc/linux/i64/cbc /tmp/be4587e282064077a143c32ddb94bf2d-pulp.mps -timeMode elapsed -branch -printingOptions all -solution /tmp/be4587e282064077a143c32ddb94bf2d-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 411 COLUMNS
At line 20981 RHS
At line 21388 BOUNDS
At line 23934 ENDATA
Problem MODEL has 406 rows, 2545 columns and 15480 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Continuous objective value is 0 - 0.14 seconds
Cgl0005I 318 SOS with 2544 members
Cgl0004I processed model has 398 rows, 2544 columns (2544 integer (2544 of which binary)) and 12936 elements
Cbc0038I Initial state - 45 integers unsatisfied sum - 10.0549
Cbc0038I Pass   1: suminf.    0.54203 (7) obj. 0 iterations 595
Cbc0038I Pass   2: suminf.    1.14934 (4) obj. 0 ite

In [2]:
from pulp import LpVariable